# NLP Text Processing Pipeline & Feature Engineering

This notebook implements the full pipeline from Data Preprocessing to Feature Engineering.

## Task 1: Preprocessing
1. Convert text to lowercase
2. Tokenization
3. Remove punctuation
4. Remove stopwords
5. Lemmatization

In [5]:

!pip install -y chromium-chromedriver
!pip install selenium pandas


[optparse.groups]Usage:[/]   
  pip install \[options] <requirement specifier> \[package-index-options] ...
  pip install \[options] -r <requirements file> \[package-index-options] ...
  pip install \[options] [-e] <vcs project url> ...
  pip install \[options] [-e] <local project path> ...
  pip install \[options] <archive url/path> ...

no such option: -y


In [6]:
!pip install selenium webdriver-manager

In [23]:
import time
import random
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--start-maximized")
options.add_argument(r"user-data-dir=C:\temp\selenium_profile")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
# 🟢 Step 1: Open Amazon manually (IMPORTANT)
driver.get("https://www.amazon.in/")
input("👉 Solve CAPTCHA/login manually if shown, then press ENTER...")

# 🔗 Step 2: Product ASIN
ASIN = "B0FLJY793G"

reviews = []
reviews = []

for page in range(1, 25):
    url = f"https://www.amazon.in/product-reviews/{ASIN}/?pageNumber={page}&sortBy=recent"
    
    driver.get(url)
    time.sleep(random.uniform(5, 8))

    print("Current URL:", driver.current_url)

    elems = driver.find_elements(By.CSS_SELECTOR, "span[data-hook='review-body']")
    
    print(f"Page {page} reviews:", len(elems))

    for e in elems:
        text = e.text.strip()
        if text:
            reviews.append(text)

    if len(reviews) >= 200:
        break

driver.quit()

# 📊 Step 4: Save ONLY review_text
df = pd.DataFrame({"review_text": reviews[:200]})
df.to_csv("amazon_reviews_200.csv", index=False, encoding="utf-8")

print(f"✅ SUCCESS: Saved {len(df)} reviews")

Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=1&sortBy=recent
Page 1 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=2&sortBy=recent
Page 2 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=3&sortBy=recent
Page 3 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=4&sortBy=recent
Page 4 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=5&sortBy=recent
Page 5 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=6&sortBy=recent
Page 6 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=7&sortBy=recent
Page 7 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=8&sortBy=recent
Page 8 reviews: 10
Current URL: https://www.amazon.in/product-reviews/B0FLJY793G/?pageNumber=9&sortBy=recent
Page 9 reviews: 10
Current URL: https:

In [24]:
df.shape

(200, 1)

In [25]:
df

,review_text
0,The product was delivered without any damage o...
1,Hot wash not working\nI have buy it one month ...
2,Nice and very use full product in budget
3,"This machine has voltage fluctuation sensor, i..."
4,I've been using this washing machine for a mon...
...,...
195,Item was working fine for last 8 months but fo...
196,"Never buy whirlpool products, they are not eas..."
197,Recently I ordered whirlpool 7.5Kg StainWash M...
198,The best. Must buy for nuclear family.


## Task 2: Vocabulary Creation
Build vocabulary manually or using sklearn. Print vocabulary size and analyze top frequent words.

In [27]:
import re
import nltk

# Download required resources (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\varik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\varik\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\varik\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [29]:
import pandas as pd

df = pd.read_csv("amazon_reviews_200.csv")

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words]
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return " ".join(tokens)

df["clean_text"] = df["review_text"].apply(preprocess)

df.to_csv("processed_reviews.csv", index=False)

In [30]:
df

,review_text,clean_text
0,The product was delivered without any damage o...,product delivered without damage scratch
1,Hot wash not working\nI have buy it one month ...,hot wash working buy one month back iam happy ...
2,Nice and very use full product in budget,nice use full product budget
3,"This machine has voltage fluctuation sensor, i...",machine voltage fluctuation sensor house volta...
4,I've been using this washing machine for a mon...,ive using washing machine month performs core ...
...,...,...
195,Item was working fine for last 8 months but fo...,item working fine last month found leakage bot...
196,"Never buy whirlpool products, they are not eas...",never buy whirlpool product easy repair anythi...
197,Recently I ordered whirlpool 7.5Kg StainWash M...,recently ordered whirlpool kg stainwash magic ...
198,The best. Must buy for nuclear family.,best must buy nuclear family


In [31]:
from sklearn.feature_extraction.text import CountVectorizer

# Load dataset
df = pd.read_csv("processed_reviews.csv")

# Create vectorizer
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(df["clean_text"])

# Vocabulary
vocab = vectorizer.get_feature_names_out()

print("Vocabulary Size:", len(vocab))

# Word frequency
import numpy as np

word_counts = X.toarray().sum(axis=0)
word_freq = dict(zip(vocab, word_counts))

# Top 10 words
top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print("\nTop 10 Frequent Words:")
for word, freq in top_words:
    print(word, ":", freq)

Vocabulary Size: 198

Top 10 Frequent Words:
product : 180
machine : 120
month : 120
good : 100
whirlpool : 100
use : 80
voltage : 80
buy : 60
delivered : 60
issue : 60


## Task 3: Feature Engineering
1. One Hot Encoding (document-level vector)
2. Bag of Words using CountVectorizer
3. TF-IDF using TfidfVectorizer

In [32]:
import pandas as pd

df = pd.read_csv("processed_reviews.csv")

# Build vocabulary
vocab = list(set(" ".join(df["clean_text"]).split()))
word_index = {word: i for i, word in enumerate(vocab)}

# One-hot encoding (document level)
one_hot_vectors = []

for text in df["clean_text"]:
    vec = [0] * len(vocab)
    for word in text.split():
        if word in word_index:
            vec[word_index[word]] = 1
    one_hot_vectors.append(vec)

print("One Hot Vector for first document:\n", one_hot_vectors[0])

One Hot Vector for first document:
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [33]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(df["clean_text"])

print("Vocabulary:", vectorizer.get_feature_names_out())
print("BoW Matrix:\n", X_bow.toarray())

Vocabulary: ['additional' 'already' 'also' 'anything' 'area' 'automatic' 'back' 'bad'
 'behavior' 'best' 'bh' 'bite' 'bottomtechnician' 'box' 'brand' 'broken'
 'budget' 'build' 'building' 'buy' 'came' 'cant' 'care' 'center' 'choose'
 'clean' 'cleaning' 'clothes' 'company' 'complete' 'concern' 'condition'
 'core' 'corrected' 'courier' 'covered' 'cup' 'customer' 'damage'
 'damaged' 'day' 'definitely' 'delay' 'delivered' 'delivery' 'demo'
 'detergent' 'disappointing' 'doesnt' 'drop' 'durable' 'easy' 'else' 'etc'
 'experience' 'exterior' 'family' 'fan' 'faulty' 'feel' 'filter' 'fine'
 'first' 'fluctuation' 'follow' 'found' 'frustrating' 'frustratingly'
 'full' 'fully' 'give' 'good' 'guided' 'happens' 'happy' 'hard' 'heater'
 'hope' 'hot' 'house' 'iam' 'idle' 'im' 'inbuilt' 'ineffective' 'inside'
 'installation' 'invest' 'issue' 'item' 'ive' 'job' 'kg' 'last' 'leakage'
 'less' 'let' 'little' 'live' 'load' 'lowquality' 'machanism' 'machine'
 'made' 'magic' 'making' 'managed' 'material' 'mesh

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(df["clean_text"])

print("TF-IDF Matrix:\n", X_tfidf.toarray())

TF-IDF Matrix:
 [[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.2812467  0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.12005715 ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.19031257 ... 0.         0.         0.47895849]]


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df['clean_text'].tolist()

# 1. One Hot Encoding (Binary CountVectorizer)
ohe_vectorizer = CountVectorizer(binary=True)
ohe_matrix = ohe_vectorizer.fit_transform(corpus)

# 2. Bag of Words (CountVectorizer standard)
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(corpus)

# 3. TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

print("Matrices Generated Successfully.\n")
print("OHE Matrix shape: ", ohe_matrix.shape)
print("BoW Matrix shape: ", bow_matrix.shape)
print("TF-IDF Matrix shape:", tfidf_matrix.shape)


Word2Vec


In [37]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')

# Load dataset
df = pd.read_csv("amazon_reviews_200.csv")

# Basic preprocessing
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', str(text))
    return word_tokenize(text)

df["tokens"] = df["review_text"].apply(preprocess)

print(df.head())

                                         review_text  \
0  The product was delivered without any damage o...   
1  Hot wash not working\nI have buy it one month ...   
2           Nice and very use full product in budget   
3  This machine has voltage fluctuation sensor, i...   
4  I've been using this washing machine for a mon...   

                                              tokens  
0  [the, product, was, delivered, without, any, d...  
1  [hot, wash, not, working, i, have, buy, it, on...  
2  [nice, and, very, use, full, product, in, budget]  
3  [this, machine, has, voltage, fluctuation, sen...  
4  [ive, been, using, this, washing, machine, for...  


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\varik\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [38]:
from gensim.models import Word2Vec

# Train Word2Vec
model = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,   # embedding size
    window=5,
    min_count=1,
    workers=4
)

# Vocabulary size
print("Vocabulary Size:", len(model.wv))

C:\Users\varik\AppData\Roaming\Python\Python310\site-packages\google\rpc\__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Vocabulary Size: 265


In [39]:
# Example: vector for a word
print(model.wv["good"])

# Similar words
print(model.wv.most_similar("good"))

[-0.10247228  0.19424638  0.06728284  0.14542073  0.01316026 -0.38134888
  0.20034409  0.58281076 -0.3491777  -0.27935576 -0.04033244 -0.28721938
 -0.16360213 -0.00760395  0.03356263 -0.20765124 -0.02399812 -0.42633313
 -0.11878681 -0.52349234  0.00159962  0.12117948  0.12632833 -0.10475519
 -0.01962947  0.02527015 -0.14651765 -0.12580086 -0.21300158  0.06736447
  0.17236677 -0.02252243  0.17279595 -0.12118968 -0.2729196   0.40261418
  0.04384451 -0.17168818 -0.16192408 -0.51283574 -0.08330498 -0.2738822
 -0.10467947  0.08787995  0.3318818  -0.13580966 -0.06701939 -0.07379825
  0.06409801  0.21971916  0.2639489  -0.09329966  0.12003585 -0.04263318
 -0.15403725  0.25609693  0.05347592  0.02028118 -0.23655516  0.13463582
  0.01278046  0.04229344  0.00317158 -0.2755585  -0.38816682  0.17016684
  0.23139024  0.37636527 -0.39581236  0.3006403  -0.16426088  0.14589319
  0.2277787  -0.09881658  0.32736576  0.2568832   0.07282346 -0.01259537
 -0.31617364  0.04378413 -0.09782575  0.11351263 -0.

## Task 4: Comparison Analysis & Task 5: Sparse Matrix Analysis

### Task 4 Comparison
| Feature | Represents | Weighting | Use Case |
|---------|------------|-----------|----------|
| **OHE** | Presence/Absence of word | Binary (0 or 1) | Simple tracking of vocabulary. |
| **BoW** | Frequency of word | Integer count | Topic modeling, naive bayes, when count matters. |
| **TF-IDF**| Importance of word | Decimal (Scale of importance) | Search engines, text classification where rare words distinguish meaning. |

**Important words in TF-IDF:** Words that appear frequently in a *single* document but rarely across the *entire corpus* receive the highest weight. 
**Why common words receive lower weight:** Common words (like "good", "the") appear in almost every document, so their Document Frequency (DF) is high. TF-IDF penalizes this, reducing their weight to near zero so the model focuses on unique, differentiating words.

### Task 5 Sparse Matrix Analysis
Sparse matrices are matrices where the vast majority of elements are zero.
*Why are they inefficient?* Storing a massive grid of mostly zeros wastes RAM and compute time. Instead, tools like Scipy use compressed formats (CSR) to store only the non-zero values and their coordinates, saving exponential amounts of memory.

In [35]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Sample data (or use your dataset)
df = pd.read_csv("processed_reviews.csv")

# Bag of Words
bow = CountVectorizer()
X_bow = bow.fit_transform(df["clean_text"])

# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df["clean_text"])

# 🔹 Shape
print("BoW Shape:", X_bow.shape)
print("TF-IDF Shape:", X_tfidf.shape)

# 🔹 Sparsity calculation
def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.count_nonzero()
    sparsity = (1 - (non_zero / total_elements)) * 100
    return sparsity

print("BoW Sparsity: {:.2f}%".format(calculate_sparsity(X_bow)))
print("TF-IDF Sparsity: {:.2f}%".format(calculate_sparsity(X_tfidf)))

BoW Shape: (200, 198)
TF-IDF Shape: (200, 198)
BoW Sparsity: 88.13%
TF-IDF Sparsity: 88.13%


## Task 6: Real-world Questions
1. **Why BoW fails in semantic meaning:** BoW completely ignores word order and context. For example, "Not good, very bad" and "Not bad, very good" have the exact same BoW vector but opposite meanings (sarcasm and negation are lost). It also treats synonyms as completely different features instead of relating them.
2. **When to use BoW vs TF-IDF:** 
   - *BoW:* Useful in basic spam detection filters, or short documents where simply mentioning a keyword is a strong indicator (e.g., ticket routing by exact keyword match).
   - *TF-IDF:* Preferred in search engine ranking algorithms, document clustering, and sentiment analysis pipeline where context-distinguishing words carry the signal.
3. **Limitations of TF-IDF:** 
   - Still suffers from the "Curse of Dimensionality" (large vocab size).
   - Cannot capture OOV (Out of Vocabulary) semantics.
   - Does not capture positional information or deep context (which modern LLMs/Word Embeddings like Word2Vec/BERT solve).